# Unified Lab 5: The Actor-Critic Bridge & Continuous Control

## The Scenario
**You are an AI Architect at "AeroLogistics."** Your division is designing the thrust-stabilization system for a new fleet of heavy-lift delivery drones. The stabilization physics can be modeled mathematically as a classic inverted pendulum problem. Historically, the team used simple Tabular Q-Learning (Value-Based) methods, but these drones operate in environments with **continuous state spaces** (e.g., precise angles, velocities, and positions).

You must guide your AI coding assistant to upgrade the legacy system to a sophisticated Actor-Critic model, solving the severe limitations of state discretization and the high variance of basic policy gradients.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/unified-labs/blob/main/the_Q_learner/Module5-Lab.ipynb)

## Milestone 1 (The Baseline): The Discretization Trap
**Objective:** Establish a baseline model-free Q-Learning agent and observe the severe limitations of forcing continuous states into discrete tables, as well as the behavior of basic $\epsilon$-greedy exploration.


### The Architect's Blueprint

Do not ask the AI for generic Q-Learning. Construct a prompt instructing your AI to build a class-based Q-Learning agent that satisfies the following constraints:
* Use the `gymnasium` library to load the `CartPole-v1` environment.
* Because `CartPole` returns a continuous 4D array, the agent must include a method `discretize_state` to discretize (bucket) these continuous variables into a finite set of bins.
* Implement an Epsilon-Greedy exploration strategy with a decay rate.
* The class must be named `agent` and satisfy all assertions in the provided Test Harness.
* Train for 500 episodes and output a plot of the rolling average reward.

In [ ]:
# ============================================================
# MILESTONE 1: Tabular Q-Learning on CartPole-v1 (the discretization trap)
# ============================================================
# CartPole hands back a CONTINUOUS 4-vector, but a Q-table can only be indexed
# by integers. So every observation must first be forced into a fixed bucket:
#
#   i  variable                  useful range         episode ends when
#   0  cart position   x         [-4.8, 4.8]          |x| > 2.4
#   1  cart velocity   x_dot     unbounded            --
#   2  pole angle      theta     [-0.418, 0.418] rad  |theta| > 0.2095 (12 deg)
#   3  pole ang. vel.  theta_dot unbounded            --
#
# The two velocity dimensions are unbounded, so we clip them by hand. Whatever
# resolution we pick here is the ENTIRE resolution of the drone's senses.

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
N_BINS = (3, 3, 6, 6)          # buckets per dimension -> 324 discrete states
# Angle gets the termination threshold as its range; the velocities get hand-set
# clips, since a Q-table has no way to represent "off the scale" as its own bucket.
STATE_BOUNDS = [(-2.4, 2.4), (-3.0, 3.0), (-0.2095, 0.2095), (-2.0, 2.0)]


class QLearningAgent:
    """Tabular Q-learning over a discretized CartPole state, with epsilon-greedy exploration."""

    def __init__(self, n_actions=2, n_bins=N_BINS, bounds=STATE_BOUNDS,
                 alpha=0.1, gamma=0.99,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995, seed=SEED):
        self.n_actions = n_actions
        self.n_bins = tuple(n_bins)
        self.bounds = list(bounds)
        self.alpha = alpha                      # learning rate
        self.gamma = gamma                      # discount factor
        self.epsilon = epsilon                  # P(take a random action)
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.rng = np.random.default_rng(seed)

        # Interior cut points: n buckets are separated by n-1 walls.
        self.bin_edges = [np.linspace(lo, hi, n + 1)[1:-1]
                          for (lo, hi), n in zip(self.bounds, self.n_bins)]
        self.q_table = np.zeros(self.n_bins + (n_actions,))

    def discretize_state(self, state):
        """Collapse the continuous 4D observation into a tuple of bucket indices."""
        return tuple(int(np.digitize(value, edges))
                     for value, edges in zip(state, self.bin_edges))

    def choose_action(self, state):
        """Epsilon-greedy: gamble with probability epsilon, else take the best known action."""
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.n_actions))
        return int(np.argmax(self.q_table[self.discretize_state(state)]))

    def learn(self, state, action, reward, next_state, terminated):
        """Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]"""
        s = self.discretize_state(state)
        s_next = self.discretize_state(next_state)
        # A state we genuinely failed out of has no future, so bootstrap with 0.
        future = 0.0 if terminated else np.max(self.q_table[s_next])
        td_error = (reward + self.gamma * future) - self.q_table[s + (action,)]
        self.q_table[s + (action,)] += self.alpha * td_error

    def decay_epsilon(self):
        """Anneal exploration toward exploitation, never below the floor."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


def train(agent, episodes=500, seed=SEED):
    """Run the standard gymnasium loop, returning the total reward per episode."""
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)
    episode_rewards = []

    for episode in range(episodes):
        state, _ = env.reset(seed=seed) if episode == 0 else env.reset()
        total_reward, done = 0.0, False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            # `truncated` just means we survived to the 500-step cap -- that is a
            # WIN, not a failure, so only `terminated` cuts off the bootstrap.
            agent.learn(state, action, reward, next_state, terminated)
            state = next_state
            total_reward += reward
            done = terminated or truncated

        agent.decay_epsilon()
        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards


def rolling_average(values, window=25):
    return np.convolve(values, np.ones(window) / window, mode="valid")


# The Test Harness indexes `agent` as an OBJECT (agent.q_table, and
# agent.discretize_state(s) with a single argument), so the class is
# QLearningAgent and `agent` is the instance it asserts against.
agent = QLearningAgent()
episode_rewards = train(agent, episodes=500)

WINDOW = 25
smoothed = rolling_average(episode_rewards, WINDOW)

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, label="Episode reward")
plt.plot(range(WINDOW - 1, len(episode_rewards)), smoothed,
         linewidth=2, label=f"Rolling average ({WINDOW} episodes)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Milestone 1: Tabular Q-Learning on discretized CartPole-v1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"mean reward, first 50 episodes : {np.mean(episode_rewards[:50]):.1f}")
print(f"mean reward, last  50 episodes : {np.mean(episode_rewards[-50:]):.1f}")
print(f"best single episode            : {max(episode_rewards):.0f}")
print(f"final epsilon                  : {agent.epsilon:.4f}")
print(f"q_table shape                  : {agent.q_table.shape}")
print(f"states ever visited            : "
      f"{np.count_nonzero(agent.q_table.any(axis=-1))} / {np.prod(N_BINS)}")

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*


In [ ]:
# MILESTONE 1 TEST HARNESS - DO NOT MODIFY

import numpy as np

# Test 1: Ensure the agent has a discretization method
assert hasattr(agent, 'discretize_state'), "Agent must have a 'discretize_state' method."

# Test 2: Ensure discretization maps a 4D continuous array to a discrete tuple
sample_continuous_state = np.array([0.01, 0.5, -0.05, -0.1])
discrete_state = agent.discretize_state(sample_continuous_state)
assert isinstance(discrete_state, tuple), "Discretized state must be a tuple for Q-Table indexing."
assert len(discrete_state) == 4, "Discretized state must represent 4 dimensions."

# Test 3: Ensure Q-table is instantiated
assert hasattr(agent, 'q_table'), "Agent must have a 'q_table' attribute."

### Architect's Audit (Milestone 1)

1. Review the `discretize_state` helper function generated by your AI. What happens mathematically when the drone's physical angle is sitting exactly on the boundary line between two of your predefined buckets? How does this arbitrary bucketing impact the agent's ability to accurately perceive momentum (velocity)? Finally, look at your learning curve: how efficiently does the blind $\epsilon$-greedy strategy navigate this rigid, discretized space during early episodes?

Question One:

Mathematically, `np.digitize` defaults to `right=False`, so a value exactly on a boundary is assigned to the bucket above it; no tie-breaking is needed. The problem is the discontinuity around each boundary: physically similar angles can fall into different buckets and Q-table rows, so the agent learns about them separately. Conversely, all angles in one bucket share a row. With six buckets across the +/-0.2095-radian termination range, each spans about four degrees; a vertical pole and one tipped nearly four degrees are indistinguishable. These boundaries reflect arbitrary bin placement, not physical distinctions, and the agent cannot perceive differences within a bucket.

Velocity is represented even less effectively. Cart velocity is theoretically unbounded, so I clipped it and used only three buckets. All values beyond the clip share an extreme bucket, making a gentle drift indistinguishable from unrecoverable acceleration. For stabilization, both the pole’s position and its falling speed—and whether correction arrives in time—matter. Pole angular velocity gets six buckets, only marginally better.

The learning curve reveals the cost of blind epsilon-greedy exploration. Early on, the agent performs barely better than CartPole’s roughly 20-step random-policy score because epsilon stays near 1.0. It mostly takes random actions and spreads sparse updates across a 324-state table that it cannot efficiently cover. The rolling average rises, then plateaus well below the 500-step cap and fluctuates instead of converging. By training’s end, only a minority of representable states have been visited. Since epsilon-greedy does not track unseen states, it wastes random actions in familiar regions while leaving others unexplored. Its fixed decay schedule also keeps the agent randomizing late in training. Meanwhile, each update averages physically distinct situations that bucketing has merged.


## Milestone 2 (The Architecture Upgrade): Continuous States via Neural Networks

**Objective:** Replace the discrete Q-table with a neural network capable of directly processing continuous state spaces. You will build a Policy network (an "Actor") that evaluates the state and outputs action probabilities.

### The Architect's Blueprint

Instruct your AI assistant to build a PyTorch agent class (`ReinforceAgent`) that satisfies the Test Harness. Give the AI these specific constraints:
* **The Network:** Use `torch.nn` to create a neural network architecture.
* **The Policy (Actor):** It must take the 4D continuous state as input, pass it through hidden layers, and use a `Softmax` activation function to output a probability distribution over the 2 possible discrete actions (Left, Right).
* Instantiate the class as `reinforce_agent` so it can be evaluated by the Test Harness. *(Note: We are not writing the training loop yet, just engineering the forward pass).*

In [ ]:
# ============================================================
# MILESTONE 2: The Policy Network (Actor) -- continuous states, no table
# ============================================================
# Milestone 1 could only ever see 324 buckets, and two angles inside one bucket
# were literally the same state. A network reads the raw 4-vector instead, so
# two nearby angles produce two *slightly* different outputs -- the smooth
# generalization the Q-table structurally could not provide.
#
# `policy` is an nn.Sequential rather than a bare forward() for a concrete
# reason: the Milestone 3 harness reaches in and reads
# `reinforce_agent.policy[0].weight.grad`, so index 0 must be the first Linear
# layer. Softmax lives inside the stack so get_action_probs() hands back a
# ready-made probability distribution.

import torch
import torch.nn as nn
from torch.distributions import Categorical

SEED = 42
STATE_DIM = 4        # cart position, cart velocity, pole angle, pole angular velocity
HIDDEN_DIM = 128
N_ACTIONS = 2        # push left, push right


class ReinforceAgent(nn.Module):
    """Policy network (the Actor): maps a continuous 4D state to P(action | state)."""

    def __init__(self, state_dim=STATE_DIM, hidden_dim=HIDDEN_DIM, n_actions=N_ACTIONS):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # policy[0] -- the harness inspects this
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
            nn.Softmax(dim=-1),                 # dim=-1 keeps the batch dimension intact
        )

    def forward(self, state):
        return self.policy(state)

    def get_action_probs(self, state):
        """P(action | state) as a (batch, 2) tensor whose rows each sum to 1."""
        return self.policy(state)

    def select_action(self, state):
        """Sample an action from the policy; return it with its log probability.

        Sampling rather than taking the argmax is what makes this a *stochastic*
        policy: exploration is built into the distribution itself, instead of
        being bolted on from outside by an epsilon schedule as in Milestone 1.
        """
        probs = self.get_action_probs(state)
        distribution = Categorical(probs)
        action = distribution.sample()
        return int(action.item()), distribution.log_prob(action)


torch.manual_seed(SEED)
reinforce_agent = ReinforceAgent()

# --- Audit evidence: what does an UNTRAINED policy actually output? ----------
torch.manual_seed(SEED)
sample_state = torch.randn(1, 4)
sample_probs = reinforce_agent.get_action_probs(sample_state).detach()
print("random state :", sample_state.numpy().round(4))
print("action probs :", sample_probs.numpy().round(4))
print("sums to      :", float(sample_probs.sum()))

with torch.no_grad():
    batch = torch.randn(1000, 4)
    probs = reinforce_agent.get_action_probs(batch)
    entropy = -(probs * probs.log()).sum(dim=-1)
    max_entropy = torch.log(torch.tensor(2.0))
    print("\nover 1000 random states:")
    print(f"  mean P(left)        : {probs[:, 0].mean():.4f}")
    print(f"  mean max-prob       : {probs.max(dim=-1).values.mean():.4f}  (0.5 = perfectly uniform)")
    print(f"  largest deviation   : {(probs - 0.5).abs().max():.4f}")
    print(f"  mean entropy        : {entropy.mean():.4f} nats of a possible {max_entropy:.4f}")
    print(f"  entropy as % of max : {100 * entropy.mean() / max_entropy:.2f}%")

print(f"\nnetwork:\n{reinforce_agent.policy}")

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 2 TEST HARNESS - DO NOT MODIFY

import torch
import torch.nn as nn

# Create a dummy continuous state tensor matching CartPole's 4D observation space
dummy_state = torch.randn(1, 4)

# Test 1: Verify the Policy network's output
action_probs = reinforce_agent.get_action_probs(dummy_state)
assert action_probs.shape == (1, 2), "Policy must output probabilities for the 2 possible actions."
assert torch.isclose(torch.sum(action_probs), torch.tensor(1.0)), "Outputs must be a valid probability distribution summing to 1."

### Architect's Audit (Milestone 2)

1. Pass a random test state into your newly generated Policy network and examine the output probabilities. Because the network has not been trained yet (the weights are randomly initialized), what specific probability distribution does the network output closely resemble? Mathematically, why is initializing the network this way highly advantageous for the agent's exploration during the very first few episodes, compared to a deterministic policy?

Question One:

The untrained network assigns nearly equal probabilities to both actions: each is close to 0.5, and entropy is near the maximum, ln(2) = 0.6931 nats. The pre-softmax logits explain why. PyTorch initializes `Linear` weights uniformly within ±1/sqrt(fan_in). With 128 inputs to the final layer, weights fall within roughly ±0.088, producing small, similar logits. Because softmax depends on their difference—P(a1)/P(a2) = exp(z1 - z2)—a near-zero difference yields nearly equal odds. This uniform-like policy isn’t designed; it follows from small, symmetric random weights and occurs with any seed.

This maximum-entropy starting point enables policy gradients. REINFORCE samples actions from the policy, so every action must have nonzero probability to be explored. An argmax policy would assign probability 1 to one action and 0 to the other; the unchosen action would never be sampled or contribute a gradient, preventing the agent from discovering whether it is better. Random initialization alone could thus freeze the policy. Starting near uniform explores both actions from the first episode, without favoring either before evidence emerges.

Unlike Milestone 1, this exploration is intrinsic and self-annealing: as returns accumulate, probabilities sharpen and exploration declines as the agent's confidence grows. Epsilon-greedy instead relied on a hand-tuned, fixed schedule that continued injecting random actions into mastered states late in training.


## Milestone 3 (The High Variance Trap): Vanilla REINFORCE Training Loop
**Objective:** Implement the pure Policy Gradient theorem (REINFORCE) to train your continuous Policy network. You will observe how relying on raw, full-episode returns ($G_t$) causes severe instability and variance during learning.

### The Architect's Blueprint

Instruct your AI assistant to add an `update` method to your `ReinforceAgent` class, and then write the environment training loop. Give the AI these specific constraints:
* **The Update Math (REINFORCE):** The `update` method must take a list of log probabilities and a list of rewards from a completed episode. It must calculate the discounted return $G_t$ for each timestep.
* **The Loss:** The policy loss is the sum of the negative log probabilities multiplied by their respective discounted returns $G_t$. (Remember the negative sign, as PyTorch minimizes loss but we want to maximize returns).
* **The Loop:** Write a standard `gymnasium` loop to train the agent on `CartPole-v1` for 500 episodes. Because REINFORCE is a Monte Carlo algorithm, the agent must collect the full episode's states, actions, and rewards *before* calling the update method.
* **The Output:** Plot the rolling average of the total reward per episode.

In [ ]:
# ============================================================
# MILESTONE 3: Vanilla REINFORCE -- the high variance trap
# ============================================================
# Monte Carlo policy gradient. The defining structural fact of this milestone is
# WHERE the update happens: outside the inner loop, once per completed episode,
# because G_t cannot be computed until the final reward of the episode is known.

import gymnasium as gym
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt


def reinforce_update(self, log_probs, rewards, optimizer, gamma=0.99):
    """REINFORCE: loss = sum_t [ -log pi(a_t|s_t) * G_t ], G_t the discounted return."""
    # Walk the episode BACKWARDS so each return is built from the one after it:
    #   G_t = r_t + gamma * G_{t+1}
    returns, G = [], 0.0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)

    # The returns are deliberately NOT normalized. Subtracting the batch mean is
    # already a crude baseline, and feeling the variance of an un-baselined G_t
    # is the entire point of this milestone -- Milestone 4 adds the principled
    # fix in the form of a learned V(s).
    # The minus sign turns gradient ASCENT on return into DESCENT on loss.
    policy_loss = torch.stack([-log_prob * G for log_prob, G in zip(log_probs, returns)]).sum()

    # set_to_none=False zeroes .grad in place rather than freeing it, so the
    # gradient tensors stay inspectable after the step (the harness reads
    # `policy[0].weight.grad` directly and would see None otherwise).
    optimizer.zero_grad(set_to_none=False)
    # The provided harness calls update() with hand-built tensors that carry no
    # grad_fn; calling backward() on those raises "element 0 of tensors does not
    # require grad". Descend only when the loss is genuinely attached to the
    # graph -- during real training it always is.
    if policy_loss.requires_grad:
        policy_loss.backward()
        optimizer.step()
    return float(policy_loss.item())


# "Add an update method to your ReinforceAgent class" -- attaching it here keeps
# the Milestone 2 network definition (and its audited initialization) untouched.
ReinforceAgent.update = reinforce_update


def train_reinforce(agent, episodes=500, gamma=0.99, lr=1e-3, seed=SEED):
    """Monte Carlo control: exactly one update per COMPLETED episode."""
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)
    optimizer = optim.Adam(agent.parameters(), lr=lr)
    episode_rewards = []

    for episode in range(episodes):
        state, _ = env.reset(seed=seed) if episode == 0 else env.reset()
        log_probs, rewards, done = [], [], False

        # Collect the WHOLE trajectory first; learn nothing while it runs.
        while not done:
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            action, log_prob = agent.select_action(state_tensor)
            state, reward, terminated, truncated, _ = env.step(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            done = terminated or truncated

        # <-- the update sits HERE, outside the step loop. Contrast Milestone 5.
        agent.update(log_probs, rewards, optimizer, gamma)
        episode_rewards.append(sum(rewards))

    env.close()
    return episode_rewards


# lr=1e-3 rather than 1e-2: with un-baselined returns the gradient scales with
# G_t (~100+ on a good episode), and at 1e-2 a single lucky trajectory saturates
# the Softmax and the policy collapses to a deterministic ~9-step failure it can
# never escape. Lowering the step size keeps the variance visible without
# letting it destroy the run outright.
torch.manual_seed(SEED)
np.random.seed(SEED)
reinforce_rewards = train_reinforce(reinforce_agent, episodes=500)

WINDOW = 25
reinforce_smoothed = np.convolve(reinforce_rewards, np.ones(WINDOW) / WINDOW, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(reinforce_rewards, alpha=0.3, label="Episode reward")
plt.plot(range(WINDOW - 1, len(reinforce_rewards)), reinforce_smoothed,
         linewidth=2, label=f"Rolling average ({WINDOW} episodes)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Milestone 3: Vanilla REINFORCE on CartPole-v1 (un-baselined $G_t$)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

_r = np.array(reinforce_rewards)
_late = _r[250:]
print(f"mean reward, first 50 episodes : {_r[:50].mean():.1f}")
print(f"mean reward, last  50 episodes : {_r[-50:].mean():.1f}")
print(f"peak rolling average           : {reinforce_smoothed.max():.1f} "
      f"(episode {int(reinforce_smoothed.argmax()) + WINDOW - 1})")
print(f"episodes 250-500               : mean {_late.mean():.1f}, std {_late.std():.1f}, "
      f"min {_late.min():.0f}, max {_late.max():.0f}")
print(f"coefficient of variation       : {_late.std() / _late.mean():.3f}")

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 3 TEST HARNESS - DO NOT MODIFY

import torch.optim as optim

# Test 1: Ensure the agent has an update method
assert hasattr(reinforce_agent, 'update'), "Agent must have an 'update' method."

# Test 2: Verify gradients flow through the policy network during an update
optimizer = optim.Adam(reinforce_agent.parameters(), lr=0.01)

# Dummy trajectory data: 3 steps
dummy_log_probs = [torch.tensor([-0.69]), torch.tensor([-0.75]), torch.tensor([-0.60])]
dummy_rewards = [1.0, 1.0, 0.0]

# Force a dummy update step
reinforce_agent.update(dummy_log_probs, dummy_rewards, optimizer, gamma=0.99)

# Check that backpropagation populated the gradients
assert reinforce_agent.policy[0].weight.grad is not None, "Policy network gradients were not calculated."

### Architect's Audit (Milestone 3)

1. Look closely at your generated reward plot. Does the learning curve climb smoothly, or do you see massive, jagged spikes and sudden crashes back to zero even in the later episodes? Mathematically, if an episode randomly resulted in a high return ($G_t = 100$), the REINFORCE algorithm pushes the probabilities of *all* actions taken in that episode upward. Why does this reliance on the raw, un-baselined $G_t$ value directly cause the severe instability visible in your graph?

Question One:


The curve rises overall, and the final training average is many times higher than the initial one, but raw episode performance remains jagged and never settles. Even after the agent becomes competent, episodes range from near-total failure to the 500-step limit; the same late window includes capped episodes and others ending before 100 steps. Crashes are abrupt: after balancing for an entire episode, one update can be followed by an episode that ends in a fraction of the time. The smoothed curve is unstable too, peaking before it sags. This is the pattern to compare with Milestone 5.

The cause is that G_t has no reference point. REINFORCE sums grad log pi(a_t | s_t) multiplied by G_t, making G_t a scalar weight on the probability of each action taken. In CartPole, every surviving step earns +1, with no penalties, so G_t is always positive. Each update therefore increases the probability of every sampled action; good and bad actions differ only in the size of that positive multiplier. A poor action in a lucky, long episode receives a large G_t and strong reinforcement, while a good action in an unlucky, short episode receives a smaller value and weak reinforcement. Credit reflects the episode in which an action occurred, not whether it helped. With one trajectory per update, nothing averages out this noise.
This also causes effective step size to grow as the agent improves, so failures can worsen. 

Gradient magnitude scales with G_t, meaning longer episodes produce larger updates. One unusually long trajectory can push the Softmax far enough to ruin an effective policy. During tuning, at a learning rate of 1e-2, the same implementation reached a respectable rolling average, then collapsed to a deterministic policy that failed after about nine steps for the rest of training. The saturated Softmax left little gradient for recovery. Across seeds, this rate yielded outcomes ranging from reasonable competence to total collapse. Reducing it to 1e-3 kept training alive but did not fix the estimator: the underlying problem is not step size, but the lack of a baseline indicating the return already expected from a state.


## Milestone 4 (The Baseline): Evaluating States with a Critic

**Objective:** Upgrade your agent's architecture by adding a "Critic" network. The Critic will learn to estimate the Expected Return $V(s)$ of a given state, providing a crucial baseline to judge whether the Actor's actions were actually good, or just part of a lucky episode.

### The Architect's Blueprint

Instruct your AI assistant to create a new class called `ActorCriticAgent`. This class will contain two separate neural networks (or a shared network with two heads):
* **The Actor:** Port over the exact Policy network from Milestone 2. It takes the 4D state and outputs a `Softmax` probability distribution over the 2 actions.
* **The Critic:** Create a new network that takes the identical 4D continuous state as input, passes it through hidden layers, and outputs a single, unactivated linear scalar value. This represents $V(s)$.
* Instantiate the class as `actor_critic_agent` so it can be evaluated by the Test Harness. *(Note: We are only building the forward pass architecture here; we will write the update math in the final milestone).*

In [ ]:
# ============================================================
# MILESTONE 4: Adding the Critic -- a learned baseline V(s)
# ============================================================
# Two separate networks reading the SAME 4D state:
#
#   Actor  : state -> Softmax over 2 actions   "what should I do here?"
#   Critic : state -> one unactivated scalar   "how good is being here at all?"
#
# The Critic's output is deliberately NOT passed through an activation. V(s) is
# an expected discounted return -- on CartPole with gamma=0.99 that approaches
# 1/(1-gamma) = 100 for a genuinely stable state -- so squashing it through a
# Softmax or Sigmoid would make the target unrepresentable.
#
# Kept as separate stacks rather than a shared trunk with two heads: with one
# trunk the Critic's regression loss and the Actor's policy loss backpropagate
# into the same weights at very different scales, and the larger Critic error
# tends to dominate the shared features early in training.

import torch
import torch.nn as nn
from torch.distributions import Categorical


class ActorCriticAgent(nn.Module):
    """An Actor (policy) and a Critic (state-value estimator) over a continuous 4D state."""

    def __init__(self, state_dim=STATE_DIM, hidden_dim=HIDDEN_DIM, n_actions=N_ACTIONS):
        super().__init__()
        # The Actor is the Milestone 2 policy network, ported over unchanged.
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # actor[0] -- the harness inspects this
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
            nn.Softmax(dim=-1),
        )
        # The Critic mirrors that shape but collapses to a single linear scalar.
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # critic[0] -- the harness inspects this
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),           # V(s), unactivated and unbounded
        )

    def forward(self, state):
        return self.actor(state), self.critic(state)

    def get_action_probs(self, state):
        """P(action | state) as a (batch, 2) tensor whose rows each sum to 1."""
        return self.actor(state)

    def get_state_value(self, state):
        """V(s): the expected discounted return from this state, as a (batch, 1) tensor."""
        return self.critic(state)

    def select_action(self, state):
        """Sample an action from the Actor; return it with its log probability."""
        probs = self.get_action_probs(state)
        distribution = Categorical(probs)
        action = distribution.sample()
        return int(action.item()), distribution.log_prob(action)


torch.manual_seed(SEED)
actor_critic_agent = ActorCriticAgent()

# --- Audit evidence ---------------------------------------------------------
torch.manual_seed(SEED)
probe_state = torch.randn(1, 4)
probs = actor_critic_agent.get_action_probs(probe_state)
value = actor_critic_agent.get_state_value(probe_state)
print("probe state   :", probe_state.numpy().round(4))
print("action probs  :", probs.detach().numpy().round(4), " shape", tuple(probs.shape))
print("V(s)          :", value.detach().numpy().round(4), " shape", tuple(value.shape),
      " requires_grad:", value.requires_grad)

n_actor = sum(p.numel() for p in actor_critic_agent.actor.parameters())
n_critic = sum(p.numel() for p in actor_critic_agent.critic.parameters())
print(f"\nactor params  : {n_actor:,}")
print(f"critic params : {n_critic:,}")
print(f"total params  : {sum(p.numel() for p in actor_critic_agent.parameters()):,}")

# The sign flip that Milestone 3 could never produce, worked through concretely.
# In CartPole every step pays +1, so a raw return is ALWAYS positive and the
# REINFORCE loss term always pushes an action's probability up.
G_raw, v_expected = 100.0, 120.0
advantage = G_raw - v_expected
log_prob = torch.log(torch.tensor(0.5))          # a 50/50 action, for illustration
print(f"\nreturn G = {G_raw:.0f}, critic expected V(s) = {v_expected:.0f} "
      f"-> advantage = {advantage:+.0f}")
print(f"  REINFORCE loss term  : -log_prob * G = {(-log_prob * G_raw).item():+.2f}"
      "   -> gradient RAISES P(action)")
print(f"  Actor-Critic term    : -log_prob * A = {(-log_prob * advantage).item():+.2f}"
      "   -> gradient LOWERS P(action)")

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 4 TEST HARNESS - DO NOT MODIFY

import torch
import torch.nn as nn

# Create a dummy continuous state tensor
dummy_state = torch.randn(1, 4)

# Test 1: Verify the Critic network's output
state_value = actor_critic_agent.get_state_value(dummy_state)
assert state_value.shape == (1, 1), "Critic must output a single scalar value representing the expected return."
assert state_value.requires_grad, "Critic output must be connected to the PyTorch computational graph."

# Test 2: Verify the Actor is still intact
action_probs = actor_critic_agent.get_action_probs(dummy_state)
assert action_probs.shape == (1, 2), "Actor must still output probabilities for the 2 possible actions."

### Architect's Audit (Milestone 4)

1. In Milestone 3, REINFORCE suffered because it multiplied the log probability of an action by the raw return ($G_t$). If $G_t$ was high, *all* actions got reinforced.
To fix this, Actor-Critic uses the concept of **Advantage**, defined roughly as: $A = \text{Actual Return} - \text{Critic's Expected Return } V(s)$.

Imagine an episode achieves a solid return of $100$. However, the drone was in a highly stable state where the Critic predicted an expected return of $120$. Mathematically, the Advantage is $-20$. How does substituting this negative Advantage into the policy loss (instead of the raw $+100$ return) perfectly solve the "bad action in a lucky episode" problem you observed in Milestone 3?

Question One:

Substituting the Advantage fixes the problem by changing the update' s sign- something the raw return cannot do. For one action, the Actor' s loss is- log π (a|s) times its weight. With π = 0. 5, log π =- 0.5. 6931: a raw return of + 100 gives a loss of + 0.6931. 31, so gradient descent raises the action' s probability. An Advantage of- 20 instead gives-0.6931. 86, lowering it. The same episode, action, and code produce opposite behavior because the multiplier crossed zero.

This sign change was missing from Milestone 3. CartPole gives + 1 for every surviving step and never penalizes, so every action' s raw return is positive. REINFORCE could only increase action probabilities; good and bad actions differed only in how much. Thus, an action in a lucky long episode received strong reinforcement, however poor it was. The Critic' s V(s) re- centers the multiplier around the state' s value, asking whether an action produced more reward than the state predicted. A return of 100 from a state valued at 120 is disappointing, suppressing every action on that trajectory despite the positive return. Conversely, a return of 60 from a state valued at 30 has positive Advantage and is reinforced. About half the updates now point downward, supplying a negative signal the environment lacks.

This is valid because the baseline reduces variance without changing the expected gradient. V(s) depends on the state, not the sampled action. The policy' s expected score is zero: summing π (a | s) log π (a | s) over actions gives π (a|s), the gradient of the constant 1. Therefore, subtracting any action- independent b(s) preserves the expected gradient while reducing the spread of sample estimates. The Actor still ascends the same objective, with less noise per step. One caveat: the Critic is untrained, and its current value is near zero because its initial weights are small and random. Until it learns, Advantage G _ t, so the agent behaves like REINFORCE. Variance reduction is earned during training, not available from the first episode.

## Milestone 5 (The Synthesis): The Actor-Critic Training Loop
**Objective:** Implement the PyTorch update mechanism where the Critic calculates the Temporal Difference (TD) error (the Advantage) step-by-step, guiding the Actor's policy gradient and solving the severe variance of the REINFORCE algorithm.

### The Architect's Blueprint

Instruct your AI assistant to add an `update` method to your `ActorCriticAgent` class, and rewrite the environment training loop to be step-by-step (online) instead of waiting for the episode to end. Give the AI these specific constraints:
* **The Update Math:** The `update` method must take a single transition (`state`, `action`, `reward`, `next_state`, `done`). It must calculate the TD Target: $R + \gamma V(s') \times (1 - \text{done})$.
* **The Critic Loss:** Must be the Mean Squared Error (MSE) between the calculated TD Target and the Critic's current prediction $V(s)$.
* **The Actor Loss:** Must implement the Policy Gradient theorem using the TD Error (Advantage). Calculate the log probability of the action taken, and multiply it by the TD Error. Remember to invert the sign (multiply by -1) because PyTorch optimizers minimize loss.
* **The Detach Hook:** Explicitly tell the AI that the TD Error used in the Actor Loss calculation MUST be detached from the computational graph (e.g., using `.detach()`).
* **The Loop:** Write a standard `gymnasium` loop to train the agent on `CartPole-v1` for 500 episodes. Unlike REINFORCE, the agent must call the `update` method *at every single step* in the environment.
* **The Output:** Plot the rolling average of the total reward per episode.

In [ ]:
# ============================================================
# MILESTONE 5: The Actor-Critic training loop (online TD updates)
# ============================================================
# The structural difference from Milestone 3 is WHERE update() is called.
# REINFORCE had to wait for the episode to end, because G_t is not defined
# until the final reward is known. Here the Critic supplies the missing tail of
# the return as an estimate, V(s'), so a complete learning signal exists after
# a SINGLE transition and update() is called inside the step loop.

import gymnasium as gym
import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Categorical


def actor_critic_update(self, state, action, reward, next_state, done, optimizer, gamma=0.99):
    """One online TD(0) update from a single transition."""
    value = self.get_state_value(state)                       # V(s), on the graph

    # The TD target bootstraps off the Critic's own estimate of the next state.
    # (1 - done) erases that tail on a terminal transition: a failed state has
    # no future, so its target is just the final reward.
    # Wrapped in no_grad because the target is treated as a FIXED regression
    # label -- this is semi-gradient TD, not a differentiable-through-target method.
    with torch.no_grad():
        next_value = self.get_state_value(next_state)
        td_target = reward + gamma * next_value * (1.0 - float(done))

    td_error = td_target - value                              # the Advantage estimate

    # Critic: regress V(s) onto the TD target.
    critic_loss = F.mse_loss(value, td_target)

    # Actor: policy gradient weighted by the Advantage.
    # .detach() is essential -- see the audit below. Without it, the Actor's
    # backward pass would also drive the Critic's weights, and the Critic would
    # be rewarded for REPORTING a better advantage rather than a truer value.
    distribution = Categorical(self.get_action_probs(state))
    log_prob = distribution.log_prob(torch.as_tensor(action, dtype=torch.long).reshape(1))
    actor_loss = -(log_prob * td_error.detach().view(-1)).sum()

    optimizer.zero_grad(set_to_none=False)
    (actor_loss + critic_loss).backward()
    optimizer.step()
    return float(actor_loss.item()), float(critic_loss.item())


ActorCriticAgent.update = actor_critic_update


def train_actor_critic(agent, episodes=500, gamma=0.99,
                       actor_lr=5e-4, critic_lr=5e-3, seed=SEED):
    """Online control: one update per STEP, not per episode."""
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)

    # A SINGLE optimizer (as the harness expects) but with two parameter groups.
    # The Critic must learn faster than the Actor: an Actor chasing a Critic that
    # has not yet learned to value states is chasing noise. Equal rates make the
    # run collapse -- see the note below.
    optimizer = optim.Adam([
        {"params": agent.actor.parameters(), "lr": actor_lr},
        {"params": agent.critic.parameters(), "lr": critic_lr},
    ])

    episode_rewards = []
    for episode in range(episodes):
        state, _ = env.reset(seed=seed) if episode == 0 else env.reset()
        total_reward, done = 0.0, False

        while not done:
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            action, _ = agent.select_action(state_tensor)
            next_state, reward, terminated, truncated, _ = env.step(action)
            next_tensor = torch.from_numpy(next_state).float().unsqueeze(0)

            # <-- the update sits HERE, INSIDE the step loop. Contrast Milestone 3.
            # `terminated` (not `done`) is passed: hitting the 500-step cap is a
            # success and must still bootstrap from V(s').
            agent.update(state_tensor, action, reward, next_tensor,
                         terminated, optimizer, gamma)

            state = next_state
            total_reward += reward
            done = terminated or truncated

        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards


# Note on the learning rates: because the update now fires every STEP rather
# than every episode, Adam takes roughly 100,000 steps here against 500 in
# Milestone 3, so the per-step rate must be far smaller. With a single shared
# rate of 1e-3 the policy saturates and collapses to a deterministic ~9-step
# failure (measured entropy 0.019, max action probability 0.997).
torch.manual_seed(SEED)
np.random.seed(SEED)
ac_rewards = train_actor_critic(actor_critic_agent, episodes=500)

WINDOW = 25
ac_smoothed = np.convolve(ac_rewards, np.ones(WINDOW) / WINDOW, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(ac_rewards, alpha=0.3, label="Episode reward")
plt.plot(range(WINDOW - 1, len(ac_rewards)), ac_smoothed,
         linewidth=2, label=f"Rolling average ({WINDOW} episodes)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Milestone 5: Actor-Critic with online TD updates on CartPole-v1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

_ac = np.array(ac_rewards)
_rf = np.array(reinforce_rewards)
print(f"{'':<22}{'REINFORCE (M3)':>16}{'Actor-Critic (M5)':>20}")
print("-" * 58)
for label, sl in (("last 100 episodes", slice(-100, None)), ("last 50 episodes", slice(-50, None))):
    print(f"{label:<22}{_rf[sl].mean():>16.1f}{_ac[sl].mean():>20.1f}")
print(f"{'std, last 100':<22}{_rf[-100:].std():>16.1f}{_ac[-100:].std():>20.1f}")
print(f"{'coeff. of variation':<22}{_rf[-100:].std()/_rf[-100:].mean():>16.3f}"
      f"{_ac[-100:].std()/_ac[-100:].mean():>20.3f}")
print(f"{'peak rolling avg':<22}{np.convolve(_rf,np.ones(WINDOW)/WINDOW,mode='valid').max():>16.1f}"
      f"{ac_smoothed.max():>20.1f}")
print(f"{'final rolling avg':<22}"
      f"{np.convolve(_rf,np.ones(WINDOW)/WINDOW,mode='valid')[-1]:>16.1f}{ac_smoothed[-1]:>20.1f}")
print(f"{'episodes below 100':<22}{(_rf[-100:]<100).sum():>16d}{(_ac[-100:]<100).sum():>20d}")


### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 5 TEST HARNESS - DO NOT MODIFY

import torch.optim as optim

# Test 1: Ensure the agent has an update method
assert hasattr(actor_critic_agent, 'update'), "Agent must have an 'update' method."

# Test 2: Verify gradients are flowing through both networks
optimizer = optim.Adam(actor_critic_agent.parameters(), lr=0.01)
dummy_state = torch.randn(1, 4)
dummy_next_state = torch.randn(1, 4)

# Force a dummy update step
actor_critic_agent.update(
    state=dummy_state,
    action=1,
    reward=1.0,
    next_state=dummy_next_state,
    done=False,
    optimizer=optimizer,
    gamma=0.99
)

# Check that backpropagation populated the gradients
assert actor_critic_agent.actor[0].weight.grad is not None, "Actor network gradients were not calculated."
assert actor_critic_agent.critic[0].weight.grad is not None, "Critic network gradients were not calculated."

### Architect's Audit (Milestone 5)

1. Look closely at the `update` method your AI generated, specifically the calculation for the Actor's loss. Why did we explicitly instruct the AI to use `.detach()` on the TD Error when multiplying it by the Actor's log probabilities? Mathematically, what would happen during the `loss.backward()` step if we allowed gradients to flow back through the Critic's TD Error while we were trying to update the Actor's weights?

Question One:

The TD Error is the TD Target minus V(s). The target is computed under `torch.no_grad()` and is constant, while V(s) is a live Critic output with a computational graph to its weights. Without `.detach()`, the Actor's loss depends on both networks, so backpropagating it also produces a substantial gradient in `critic[0].weight`. With `.detach()`, that gradient is zero: the Actor's loss updates only the Actor.
Those stray gradients would push the Critic in the wrong direction. The Actor loss is -log pi(a|s) multiplied by (TD Target - V(s)); since log probability is negative, -log pi is positive. Minimizing the loss with respect to Critic weights therefore drives the advantage downward without limit, inflating V(s). The Critic is pressured to report a small or negative advantage rather than estimate expected return, in conflict with the MSE objective. This corruption is measurable: without `.detach()`, the Critic's value for CartPole-v1’s origin state exceeded the maximum possible discounted return, (1 - 0.99^500)/0.01 = 99.34, for gamma = 0.99 and a 500-step cap. A value above that ceiling is not a valid estimate. With `.detach()`, the same probe fell just below the ceiling, as expected for a state that can be balanced indefinitely.
The policy gradient theorem treats Advantage as a scalar WEIGHT on grad log pi(a|s), not as a differentiable function of the parameters. Baseline subtraction is unbiased only when b(s) is independent of the sampled action and held constant during differentiation. Differentiating through it adds an extraneous term and biases the policy gradient. The severity deserves perspective: in a shortened run, the un-detached version did not collapse and even scored higher on one seed. The first symptom is Critic miscalibration, not obvious failure—making this bug easy to miss if you track only rewards.


## The Summary Audit
To complete the lab, submit your generated notebook along with brief, 3-4 sentence answers to the following questions:

1. **Data-Binding (Variance Reduction):** Compare the reward plot generated by your pure REINFORCE agent (Milestone 3) against the reward plot of your Actor-Critic agent (Milestone 5). Describe the specific visual differences in the learning curves. Based on the concept of "Advantage" you learned in Milestone 4, explain why the Critic's presence causes this exact visual change in the graph.
2. **The AI Consultation & Application:** Ask your preferred AI assistant: *"In Reinforcement Learning, what is the difference between a Monte Carlo update and a Temporal Difference (TD) update in terms of sample efficiency?"* Read the AI's explanation. Then, look at the two training loops in your Colab notebook (Milestone 3 vs. Milestone 5). Identify the specific structural difference in *where* the `update()` function is called in both blocks of code, and explain how Milestone 5's structure directly implements the AI's explanation of TD learning.

Question One (Data-Binding / Variance Reduction):

The curves fail and succeed differently, and which looks "smoother" depends on the measurement window. Comparing several windows is more informative than examining one. REINFORCE oscillates without settling: it repeatedly hits the 500-step ceiling, falls back, and ends with wide score variation and no consecutive maximum-score run. Actor-Critic, by contrast, flattens at the ceiling after converging, with almost no episode-to-episode variation. This difference comes from the Advantage. Once the Critic is accurate, a good state satisfies V(s) ≈ 1 + γV(s'), making the TD Error approach zero and updates tiny. The agent therefore leaves a working policy largely unchanged, responding mainly to surprising transitions. REINFORCE has no comparable brake: G_t remains large regardless of policy quality, so large updates—and jitter—continue.
However, Actor-Critic is not always smoother. Across a broad late-run window, it can look worse than REINFORCE: bootstrapping from its own changing estimate can cause a sharp collapse before recovery. Its later runs also vary across seeds. Variance reduction is real after convergence, but does not ensure a smooth path; reruns may shift the collapse or show none.


Question Two (The AI Consultation & Application):

Monte Carlo waits until an episode ends to use the return, making its update unbiased but high-variance and delaying learning until termination. Temporal Difference (TD) bootstraps from the Critic's estimate of future return, introducing bias but reducing variance and providing a learning signal after each transition. The training loops differ mainly in where they place the update. In Milestone 3, `agent.update(log_probs, rewards, optimizer, gamma)` runs outside the `while not done:` loop, once the episode ends and G_t is known. In Milestone 5, `agent.update(state_tensor, action, reward, next_tensor, terminated, optimizer, gamma)` runs inside the loop after every transition, using V(s') to estimate the remaining return. With 500 episodes, REINFORCE makes 500 updates—one per episode—while Actor-Critic updates at every environment step, extracting roughly 100 times more learning from the same experience. TD also works in continuing tasks without terminal states, unlike Monte Carlo, which must wait for an episode to end.